# Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils import shuffle
from transformers import AutoModel, BertTokenizerFast
from transformers import BatchEncoding
from tokenizers import Encoding
import pickle
from matplotlib import pyplot as plt
import os
import text_preprocess as tp
import torch
from tensorflow.keras.models import load_model

import ast

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

This note book generate features for randomly sampled 200 sentences from EBM-NLP data:

- Word Embedding
- Section (ignored for EBM-NLP dataset)
- Fragment
- Split into train/evaluation

In [ ]:
# Load preprocessing functions
processor = tp.Text_Processor(sent_len = 100, \
                             max_words_per_sec = 500, \
                              max_sent_per_sec = 20, \
                              sent_len_per_sec = 40, \
                              emb_dim = 768)

In [ ]:
if torch.cuda.is_available():    

    # Tell PyTorch to use the GPU.    
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

## Load Data and BERT Models

In [ ]:
train_data_dir = '../data_full/train/200/0'
test_data_dir = '../data_full/test'
sampled_train_df = pd.read_csv(os.path.join(train_data_dir, 'train_sentence.csv' ))
test_df = pd.read_csv(os.path.join(test_data_dir, 'test_sentence_expert.csv' ))

sampled_train_df.head()

In [ ]:
# Load the BERT tokenizer.
print('Loading BERT tokenizer...')

tokenizer = BertTokenizerFast.from_pretrained("dmis-lab/biobert-base-cased-v1.1")

bert_model = AutoModel.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    #num_labels = num_labels, 
    output_attentions = False, # Whether the model returns attentions weights.
    output_hidden_states = True, # Whether the model returns all hidden-states.
        
    )
#bert_model.cuda()

In [ ]:
bert_model.to(device)

# Sentence Word Embedding

In [ ]:
df_dict = {"train": sampled_train_df, "test": test_df}

sent_tokens = {}
for key in df_dict:
    print(f"========={key}=========")
    tokens = [tokenizer.tokenize(s) for s in df_dict[key].text.values]
    lens = [len(item) for item in tokens]
    pd.Series(lens).hist()
    plt.show()
    sent_tokens[key] = tokens


In [ ]:
for key in df_dict:
    
    print(f"========={key}=========")
    
    #sent_syn = []
    annotations = []
    sentences = df_dict[key].text.values
    for s in df_dict[key].text.values:
      tokens, syn, annotation = processor.tokenizer(s)
      #sent_syn.append(syn)
      annotations.append(annotation)
        
    #syn_vector = processor.vectorize_syn(sent_syn)
    
    token_embeddings, alignment, attention_masks = processor.get_bert_emb_and_alignment(sentences, tokenizer, bert_model, annotations, device, max_length = processor.sent_len)

    print(f"embedding size: {token_embeddings.shape}, mask size: {attention_masks.shape}")
    
    if key == 'train':
      data_dir = train_data_dir
    else:
      data_dir = test_data_dir
    
    np.save( os.path.join(data_dir, "wordvectors.npy"),token_embeddings)
    np.save(os.path.join(data_dir, "sentence_masks.npy"), attention_masks)
    
    pickle.dump(alignment, open(os.path.join(data_dir, "alignment.pkl"), 'wb'))


# Section Embedding (Not needed for EBM-NLP dataset)

In [ ]:
# Load section model

data_path = '../data/train/200/0'

section_model = load_model('../section_model')
section_model.summary()


In [ ]:
section_prob = []
section_encoding = []
for section in df.section.values:
  sec_wv = processor.process_section_bert(section, tokenizer, bert_model, device, \
                                          sent_len=processor.sent_len_per_sec, \
                                          max_sent = processor.max_sent_per_sec, emb_size = 768)

  prob = section_model.predict(sec_wv[None,:])

  section_prob.append(prob.reshape(-1))
  #section_encoding.append(lstm_code[0])

section_prob = np.array(section_prob)
print(section_prob.shape)


#np.save("../data/section_encoding.npy", section_encoding)

In [ ]:
np.save(os.path.join(data_path,"masked", "section_prob.npy"),section_prob)
np.save(os.path.join(data_path, "unmasked","section_prob.npy"),section_prob)

In [ ]:
i = 5
section_prob[i]
df.iloc[i]["section"]

# Fragments

In [ ]:
def get_fragment_mask(frag, frag_inds, sent, tokenizer,sent_len = 100):

  if frag is not None:

    tokenized_batch : BatchEncoding = tokenizer(sent, truncation=True,
                                add_special_tokens = True, # Add '[CLS]' and '[SEP]'
                                max_length = sent_len,           # Pad & truncate all sentences.
                                pad_to_max_length = True,
                                return_attention_mask = True,   # Construct attn. masks.
                                return_tensors = 'pt')
            
    tokenized_text :Encoding  =tokenized_batch[0]
    tokens = tokenized_text.tokens

    fs = frag.split(";")
    ids = ast.literal_eval(frag_inds)

    frag_tokens = []
    frag_tokens_ids = []

    for i, f in zip(ids, fs):
      
      for idx in range(int(i), int(i)+len(f)):
        token_id = tokenized_text.char_to_token(idx)
        if token_id is not None:
          if token_id not in frag_tokens_ids:
            frag_tokens_ids.append(token_id)
            frag_tokens.append(tokens[token_id])

    #print(f, frag_tokens_ids, frag_tokens)
    return frag_tokens,frag_tokens_ids


In [ ]:
LABEL_DECODERS =  { \
      'participants': { \
        0: 'No label',
        1: 'Age',
        2: 'Sex',
        3: 'Sample-size',
        4: 'Condition' },

      'interventions': { \
        0: 'No label',
        1: 'Surgical',
        2: 'Physical',
        3: 'Pharmacological',
        4: 'Educational',
        5: 'Psychological',
        6: 'Other',
        7: 'Control' },

      'outcomes': { \
        0: 'No label',
        1: 'Physical',
        2: 'Pain',
        3: 'Mortality',
        4: 'Adverse-effects',
        5: 'Mental',
        6: 'Other' }
    }

lower_labels = {k1: [k1+"/"+LABEL_DECODERS[k1][k2] for k2 in LABEL_DECODERS[k1] if k2!=0] for k1 in LABEL_DECODERS } 
lower_labels

In [ ]:
df_dict["train"].iloc[2].T

In [ ]:
file_path_dict = {"train": '../data_full/train/200/0', 
                  'test':'../data_full/test'}

for key in df_dict:
    print(f"========{key}=========")
    sent_len = 100

    all_frag_token_ids = {}
    all_frag_tokens = {}

    df = df_dict[key]
    attention_masks = np.load(os.path.join(file_path_dict[key],"sentence_masks.npy"))
    
    for col in ['participants', 'interventions', 'outcomes']:
      
      # initial fragment contains all tokens
      # if a sentence does not belong to this class, the fragment consists of all tokens
      frag_token_ids = attention_masks.copy() # use a copy, otherwise attention_masks will be modified
      frag_tokens = [[]]*len(attention_masks)

      for idx, row in df[df[col]==1].iterrows():
          
          # reset mask to all zeros
          frag_token_ids[idx] = 0

          # combine all fragments

          for sub_label in lower_labels[col]:
              
            if not pd.isnull(row[sub_label+' Fragment']): 
                tokens,tokens_ids = get_fragment_mask(row[sub_label+' Fragment'], row[sub_label+'_fragment_index'], 
                              row['text'], tokenizer)
                
                # update the mask to only fragment
                frag_token_ids[idx][tokens_ids] = 1
                frag_tokens[idx] = frag_tokens[idx] + tokens
                
      all_frag_token_ids[col] = frag_token_ids
      all_frag_tokens[col] = frag_tokens
        
    pickle.dump(all_frag_token_ids, open(os.path.join(file_path_dict[key], "frag_token_ids.pkl"), 'wb'))
    pickle.dump(all_frag_tokens, open(os.path.join(file_path_dict[key], "frag_tokens.pkl"), 'wb'))   

In [ ]:
# granular fragments

for key in df_dict:
    print(f"========{key}=========")
    sent_len = 100

    all_frag_token_ids = {}
    all_frag_tokens = {}

    df = df_dict[key]
    attention_masks = np.load(os.path.join(file_path_dict[key], "sentence_masks.npy"))
    
    all_labels = []
    for l in lower_labels:
        all_labels += lower_labels[l]
        
    for col in all_labels:
      
      # initial fragment contains all tokens
        frag_token_ids = attention_masks.copy() # use a copy, otherwise attention_masks will be modified
        frag_tokens = [[]]*len(attention_masks)

        for idx, row in df[df[col]==1].iterrows():
          
            # reset mask to all zeros
            frag_token_ids[idx] = 0

            tokens,tokens_ids = get_fragment_mask(row[col+' Fragment'], row[col+'_fragment_index'], 
                              row['text'], tokenizer)
                
            # update the mask to only fragment
            frag_token_ids[idx][tokens_ids] = 1
            frag_tokens[idx] = tokens
                
        all_frag_token_ids[col] = frag_token_ids
        all_frag_tokens[col] = frag_tokens
        
    pickle.dump(all_frag_token_ids, open(os.path.join(file_path_dict[key], "granular_frag_token_ids.pkl"), 'wb'))
    pickle.dump(all_frag_tokens, open(os.path.join(file_path_dict[key], "granular_frag_tokens.pkl"), 'wb'))   

# Split into Train/Test

In [ ]:
# Get train and validation *data* for each aspect
# will just save the index of these sentences

def train_val_split(df, label_cols, fold = 5):

    #splite train (2/3) and validation (1/3)
    frac = (fold - 1)/fold

    # sentence indexes for train 
    Train_pos_set ={}
    Train_neg_set ={}

    # sentence indexes for validation validation
    Val_pos_set ={}
    Val_neg_set ={}

    for name in label_cols:
      if name!='Negative':
          print("\n", name)
          X_train=[]
          X_val=[]

          # positive case
          x = df[df[name]==1][name].index.values
          x = shuffle(x, random_state=42)
          N = int(len(x)*frac)
          pos_train = x[0:N]  # Only sentence ids are stored
          pos_val = x[N:]
          
          print("pos_train", len(pos_train))
          print("pos_val", len(pos_val))
          
          # Negative case
          # double sample the cases for the other classes
          other_class = df[(df[name]==0) & (df["Negative"]==0)][name].index.values 
          neg = df[df["Negative"]==1][name].index.values  

          neg = np.concatenate([other_class, neg], axis = 0)
          
          neg = shuffle(neg, random_state=42)

          #split train and validation data in negative case
          N = int(len(neg)*frac)
          neg_train = neg[0:N]
          neg_val = neg[N:]
          print("neg_train", len(neg_train))
          print("neg_val", len(neg_val))
          
          # store the indexes into a diction
          Train_pos_set[name] = pos_train
          Train_neg_set[name] = neg_train

          Val_pos_set[name] = pos_val
          Val_neg_set[name] = neg_val
    
    return Train_pos_set, Train_neg_set, Val_pos_set, Val_neg_set

In [ ]:
sampled_train_df = pd.read_csv(os.path.join(data_path, "train_sentence.csv"))

In [ ]:
label_cols = ['participants','interventions','outcomes','Negative']
sampled_train_df[label_cols] = sampled_train_df[label_cols].fillna(0)
sampled_train_df[label_cols].sum(0)

In [ ]:

Train_pos_set, Train_neg_set, Val_pos_set, Val_neg_set = train_val_split(sampled_train_df, label_cols, fold = 5)

pickle.dump(Train_pos_set, open(os.path.join(file_path_dict['train'], "train_pos_dict.pkl"),"wb"))
pickle.dump(Val_pos_set, open(os.path.join(file_path_dict['train'],"val_pos_dict.pkl"),"wb"))
pickle.dump(Train_neg_set, open(os.path.join(file_path_dict['train'],"train_neg_dict.pkl"),"wb"))
pickle.dump(Val_neg_set, open(os.path.join(file_path_dict['train'], "val_neg_dict.pkl"),"wb"))

